In [1]:
import numpy as np
import astroemperor as emp

np.random.seed(1234)

I couldnt grab the terminal size. Trying with pandas...
Terminal size with pandas successful!


## engine setup

In [2]:
sim = emp.Simulation()
sim.load_data('51Peg')  # folder read from /datafiles/

sim.set_engine('reddemcee')
#sim.engine_config['setup'] = [8, 128, 512, 1]  # ntemps, nwalkers, nsweeps, nsteps
sim.engine_config['setup'] = [8, 64, 256, 1]  # ntemps, nwalkers, nsweeps, nsteps


sim.run_config['logger_level'] = 'CRITICAL'
#sim.cores__ = 12  # threads to use

                                                                                
                   ~~ Simulation Successfully Initialized ~~                    
                                                                                


                         
Reading data from 51peg.vels                          








## model setup

In [3]:
sim.instrument_names_RV = ['LICK']
sim.starmass = 1.12
sim.keplerian_parameterisation = 1

#sim.debug.debugging = True
sim.add_condition(['Period 1', 'limits', [3, 5]])
sim.add_condition(['Amplitude 1', 'limits', [45, 60]])

sim.add_condition(['Offset 1', 'limits', [-10., 10.]])

sim.add_condition(['Period 1', 'init_pos', [4.1, 4.3]])
sim.add_condition(['Amplitude 1', 'init_pos', [50, 60]])

## Plot options

In [4]:
sim.plot_posteriors['temps'] = [0]
sim.plot_trace['plot'] = False
sim.plot_histograms['plot'] = False

In [5]:
sim.autorun(1, 1)

                              Offset block added, OffsetBlock


                              Jitter block added, JitterBlock


                              Keplerian block added, KeplerianBlock 1



Condition applied: Parameter Period 1 attribute limits set to [3, 5]

Condition applied: Parameter Period 1 attribute init_pos set to [4.1, 4.3]

Condition applied: Parameter Amplitude 1 attribute limits set to [45, 60]

Condition applied: Parameter Amplitude 1 attribute init_pos set to [50, 60]

Condition applied: Parameter Offset 1 attribute limits set to [-10.0, 10.0]


                                ~~ Setup Info ~~                                


Current Engine is            reddemcee 0.9.12
Number of cores is           24
Save location is             datalogs/51Peg/run_17/k1
Dynamical Criteria is        None
Posterior fit method is      Gaussian Mixtures
Limits constrain method is   range
Model Selection method is    BIC


                           ~~ Automatically Saving ~~    

100%|██████████| 2048/2048 [00:11<00:00, 185.57it/s]


temp_script.py took 11.096 seconds
Autocorrelation tolerance=50 fails. Setting to 0.
                         Calculating Gaussian Mixtures                          


100%|██████████| 11/11 [00:00<00:00, 22.37it/s]




                                 ~~ Best Fit ~~                                 


Parameter        Value(max)  Range(±σ)           Prior               Limits
-------------  ------------  ------------------  ------------------  -------------
Period 1              4.231  [4.231, 4.231]      ~𝓤 (3, 5)           [3, 5]
Amplitude 1          55.635  [53.942, 57.056]    ~𝓤 (45, 60)         [45, 60]
Phase 1               2.468  [1.541, 3.168]      ~𝓤 (0.0, 6.283)     [0, 6.283]
Ecc_sin 1             0.038  [-0.064, 0.206]     ~𝓤 (-1, 1)          [-1, 1]
Ecc_cos 1             0.14   [0.131, 0.191]      ~𝓤 (-1, 1)          [-1, 1]

-------------  ------------  ------------------  ------------------  -------------
Offset 1              3.608  [3.518, 4.541]      ~𝓤 (-10.0, 10.0)    [-10, 10]

-------------  ------------  ------------------  ------------------  -------------
Jitter 1              3.563  [3.15, 6.787]       ~𝓝 (5, 5)           [0, 75.852]




                                 ~~ 

...



                           Plotting Temperature Rates                           


100%|██████████| 7/7 [00:00<00:00, 10.98it/s]




                           Plotting Gaussian Mixtures                           


100%|██████████| 11/11 [00:02<00:00,  4.40it/s]





Time Table
Time RUN                   : 00:00:13
Time POSTPROCESS           : 00:00:00
Time CALCULATE GM          : 00:00:00
Time plot_posteriors       : 00:00:05
Time plot_histograms       : 00:00:00
Time plot_keplerian_model  : 00:00:00
Time plot_betas            : 00:00:00
Time plot_beta_density     : 00:00:00
Time plot_rates            : 00:00:00
Time plot_trace            : 00:00:00
Time plot_astrometric_model: 00:00:00

(Past BIC) - (Present BIC) > 5

 BIC condition met!!
inf - 1780.749 > 5


                                                                                
                              ~~ End of the Run ~~                              
                                                                                




In [6]:
sim.sampler.get_evidence_hybrid(discard=sim.reddemcee_discard)

(-2561.071219915976, 9.784431057226941)

In [12]:
sim.sampler.get_evidence_ti(discard=sim.reddemcee_discard)

(-1113.9879927056386, 11.868692882491823)

In [7]:
sim.sampler.get_evidence_ss(discard=0)

(-3332.6751576338297, 3.1652136157942574)

In [ ]:
asd

In [8]:
discard = sim.reddemcee_discard
L = sim.sampler.get_log_like(discard=discard)  # shape=(ntemps, nsweeps, nwalkers)
B = sim.sampler.get_betas(discard=discard)  # shape=(ntemps, nsweeps)


In [9]:
ntemps, nsweeps = B.shape

m = ntemps-1



In [10]:
B.shape

(8, 256)

In [11]:
logA_series = np.empty((nsweeps, m), dtype=float)
logC_series = np.empty((nsweeps, m), dtype=float)

delta = (B[1:, :].mean(axis=1) - B[:-1, :].mean(axis=1))  # (m,)
half = 0.5 * delta
for i in range(m):
    # per-sweep log-mean-exp across walkers delta-beta i, i+1
    sA = half[i] * L[i, :, :]  # (nsweeps, nwalkers)
    sC = -half[i] * L[i+1, :, :]
    logA_series[:, i] = _logsumexp(sA, axis=1) - np.log(sA.shape[1])
    logC_series[:, i] = _logsumexp(sC, axis=1) - np.log(sC.shape[1])

logmuA = _logsumexp(logA_series, axis=0) - np.log(nsweeps)  # (m,)
logmuC = _logsumexp(logC_series, axis=0) - np.log(nsweeps)


shiftA = np.max(logA_series, axis=0)  # (m,)
shiftC = np.max(logC_series, axis=0)

A_scaled = np.exp(logA_series - shiftA)  # <= 1, no overflow
C_scaled = np.exp(logC_series - shiftC)

muA_scaled = A_scaled.mean(axis=0)  # corresponds to muA / exp(shiftA)
muC_scaled = C_scaled.mean(axis=0)

z_hat = float(np.sum(logmuA - logmuC))
W_series = np.hstack([A_scaled, C_scaled])  # (nsweeps, 2m)
grad = np.concatenate([1.0 / muA_scaled, -1.0 / muC_scaled])



NameError: name '_logsumexp' is not defined

In [ ]:
logA_series0

array([[964.50952951, 359.32082907, 773.36904417, ..., 239.23126318,
         78.99098946,  22.96045909],
       [164.02518022, 172.57180414, 193.22124237, ..., 210.39522267,
         52.74236541,  29.30575609],
       [163.99095161, 154.62250374, 149.24515712, ...,  88.81091071,
         58.34906347,  29.30184901],
       ...,
       [133.90148291, 127.095514  , 121.77346125, ...,  53.35395847,
         16.22894889,   6.00357513],
       [133.90148291, 127.095514  , 122.53810865, ...,  53.2739547 ,
         16.27283601,   6.0068075 ],
       [133.90148291, 127.095514  , 122.07580835, ...,  53.23794816,
         16.36534366,   6.05923245]])

In [ ]:
def _logsumexp(a, axis=None, keepdims=False):
    # more numerical stability
    a = np.asarray(a, dtype=float)
    amax = np.max(a, axis=axis, keepdims=True)
    # If everything is -inf along an axis, keep it stable
    out = amax + np.log(np.sum(np.exp(a - amax), axis=axis, keepdims=True))
    if not keepdims and axis is not None:
        out = np.squeeze(out, axis=axis)
    return out

In [ ]:
def _logmeanexp(x, axis=None):
    # numerical stability
    a = np.max(x, axis=axis, keepdims=True)
    return (a + np.log(np.mean(np.exp(x - a), axis=axis, keepdims=True))).squeeze()


array([[2.44133686e-66, 5.50316339e-65, 2.51700802e-57, ...,
        5.69851181e-27, 1.03638600e-08, 1.31145766e-04],
       [1.63858666e-67, 1.94553343e-65, 5.09789341e-59, ...,
        5.20004944e-28, 1.88916700e-09, 3.90311817e-04],
       [1.77638353e-66, 6.77687793e-60, 7.00081205e-61, ...,
        1.44211109e-27, 1.24842583e-09, 3.02078232e-04],
       ...,
       [1.99237947e-46, 2.77349074e-44, 5.65562521e-43, ...,
        3.25894931e-22, 4.23748036e-08, 5.17147594e-04],
       [1.86870963e-46, 3.05512979e-44, 6.73873809e-43, ...,
        3.57612902e-22, 4.23192331e-08, 4.81960708e-04],
       [1.90126688e-46, 3.29795611e-44, 5.92839172e-43, ...,
        4.54216988e-22, 3.79006330e-08, 5.39255691e-04]])

In [ ]:
reddemcee.utils._ss_bridge(B, L, True)

/IronCrane/reddtea/pip_packages/reddemcee/src/reddemcee/utils.py:293: RuntimeWarning: overflow encountered in exp
  A_series = np.exp(logA_series)  # (nsweeps, m)


(inf,
 array([[            inf, 1.12474300e+156,             inf, ...,
         5.69851181e-027, 1.03638600e-008, 1.31145766e-004],
        [1.71882104e+071, 8.85079476e+074, 8.22089965e+083, ...,
         5.20004944e-028, 1.88916700e-009, 3.90311817e-004],
        [1.66098369e+071, 1.41807810e+067, 6.55161225e+064, ...,
         1.44211109e-027, 1.24842583e-009, 3.02078232e-004],
        ...,
        [1.42126529e+058, 1.57354948e+055, 7.68320221e+052, ...,
         3.25894931e-022, 4.23748036e-008, 5.17147594e-004],
        [1.42126529e+058, 1.57354948e+055, 1.65053376e+053, ...,
         3.57612902e-022, 4.23192331e-008, 4.81960708e-004],
        [1.42126529e+058, 1.57354948e+055, 1.03956090e+053, ...,
         4.54216988e-022, 3.79006330e-008, 5.39255691e-004]]),
 array([ 0.00000000e+000,  2.27607551e-154,  0.00000000e+000,
         1.28127176e-297,  3.24655277e-102,  1.26732826e-032,
         2.19269208e-011, -4.07581428e+046, -4.22599298e+044,
        -3.09684464e+043, -3.81441287

In [ ]:
np.sum(np.diff(B, axis=0)>0)

0